# Can we predict failure within 90 days of an inspection?
Target: `failed_90d`. Plan: inspect data, check for leakage, build baselines, evaluate on a **time-ordered holdout** (train on earlier inspections, test on the latest), and report discrimination, calibration and a usable operating point.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, brier_score_loss
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
pd.set_option('display.width',150)
df=pd.read_csv('data.csv',parse_dates=['inspected_at']).sort_values('inspected_at').reset_index(drop=True)
display(pd.read_csv('columns.csv'))
print(df.shape, 'missing:',int(df.isna().sum().sum()),'dupes:',int(df.duplicated().sum()),'unique assets:',df.asset_tag.nunique())
print('date range',df.inspected_at.min(),'->',df.inspected_at.max())
print('base rate',df.failed_90d.mean().round(4))
df.describe().T

## Leakage check: `parts_replaced`
The dictionary says it is the parts replaced on the work order that *closed* the inspection record, i.e. recorded after the inspection and plausibly after a failure/repair. It would not be known at inspection time.

In [ ]:
display(df.groupby('parts_replaced').failed_90d.agg(['mean','size']))

Zero parts replaced gives 0% failures; two or more gives 100%. That near-deterministic relationship is the signature of leakage (the work order reflects the outcome), so **`parts_replaced` is excluded**. A model using it would look near-perfect and be useless at inspection time. `asset_tag` (all unique) is an identifier and is dropped too.

## Exploration of legitimate features

In [ ]:
num=['vibration_rms','bearing_temp_c','ambient_temp_c','meter_hours','load_pct']
fig,ax=plt.subplots(1,5,figsize=(18,3.2))
for a,f in zip(ax,num):
    for k,g in df.groupby('failed_90d'): a.hist(g[f],bins=30,alpha=.55,density=True,label=f'failed={k}')
    a.set_title(f)
ax[0].legend(); plt.tight_layout(); plt.show()
print(df.groupby('failed_90d')[num].mean().T)
print(df.groupby('site').failed_90d.agg(['mean','size']))
print(df.groupby(df.inspected_at.dt.to_period('Q')).failed_90d.agg(['mean','size']))
print('single-feature AUC:', {f:round(roc_auc_score(df.failed_90d,df[f]),3) for f in num})

## Drift: the failure rate is not stable over time
Monthly failure rate swings from ~0% to ~39%, and `meter_hours` is essentially a calendar proxy (the average climbs steadily across quarters, since each asset is inspected once). The strength of individual features also changes between quarters. So a random shuffled CV would mix time periods and be optimistic about how the model performs on *future* inspections.

In [ ]:
m=df.groupby(df.inspected_at.dt.to_period('M')).failed_90d.mean()
m.plot(kind='bar',figsize=(12,2.8),title='Failure rate by inspection month'); plt.show()
q=df.assign(q=df.inspected_at.dt.to_period('Q'))
print(q.groupby('q')[num].mean().round(2))
print('within-quarter AUC per feature'); print(q.groupby('q').apply(lambda g: pd.Series({f: roc_auc_score(g.failed_90d,g[f]) for f in num})).round(2))

## Time-ordered split
The last 20% of inspections (by date) form the test set; nothing from it is used for fitting or model selection. Quarterly failure rates above also show whether late inspections have an incomplete 90-day follow-up window.

In [ ]:
feats=num+['site']
cut=int(len(df)*.8)
tr,te=df.iloc[:cut],df.iloc[cut:]
Xtr,ytr,Xte,yte=tr[feats],tr.failed_90d,te[feats],te.failed_90d
print('train ends',tr.inspected_at.max(),'| test starts',te.inspected_at.min(),'| rates',ytr.mean().round(3),yte.mean().round(3))
models={'baseline (prevalence)':DummyClassifier(strategy='prior'),
 'logistic regression':make_pipeline(ColumnTransformer([('n',StandardScaler(),num),('c',OneHotEncoder(),['site'])]),LogisticRegression(max_iter=1000)),
 'gradient boosting':make_pipeline(ColumnTransformer([('n','passthrough',num),('c',OneHotEncoder(),['site'])]),
     HistGradientBoostingClassifier(max_depth=3,learning_rate=.05,max_iter=200,l2_regularization=1.0,random_state=0))}
rows=[];probs={}
cv=list(TimeSeriesSplit(5).split(Xtr))
for n,m in models.items():
    # expanding-window CV inside the training period only
    vi=np.concatenate([v for _,v in cv])
    p_cv=np.concatenate([m.fit(Xtr.iloc[t],ytr.iloc[t]).predict_proba(Xtr.iloc[v])[:,1] for t,v in cv])
    m.fit(Xtr,ytr); p=m.predict_proba(Xte)[:,1]; probs[n]=p
    rows.append(dict(model=n,cv_auc=roc_auc_score(ytr.iloc[vi],p_cv),test_auc=roc_auc_score(yte,p),test_pr_auc=average_precision_score(yte,p),test_brier=brier_score_loss(yte,p)))
res=pd.DataFrame(rows).set_index('model').round(3); display(res)
print('test base rate (= PR-AUC of a random model):',yte.mean().round(3), '| n test',len(yte),'| failures',int(yte.sum()))

## For comparison: shuffled (non-temporal) CV on all data

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
for nm in ['logistic regression','gradient boosting']:
    pp=cross_val_predict(models[nm],df[feats],df.failed_90d,cv=StratifiedKFold(5,shuffle=True,random_state=0),method='predict_proba')[:,1]
    print(nm,'shuffled 5-fold AUC',round(roc_auc_score(df.failed_90d,pp),3))

## Bootstrap uncertainty on the test set

In [ ]:
rng=np.random.default_rng(0); y=yte.values
for nm in ['logistic regression','gradient boosting']:
    pp=probs[nm]; b=[roc_auc_score(y[i],pp[i]) for i in (rng.integers(0,len(y),len(y)) for _ in range(500))]
    print(nm,'test AUC 95% CI',np.percentile(b,[2.5,97.5]).round(3))

## Calibration, operating point and drivers

In [ ]:
best=max(['logistic regression','gradient boosting'],key=lambda n:res.loc[n,'test_pr_auc']); p=probs[best]; print('best model:',best)
fig,ax=plt.subplots(1,3,figsize=(16,4))
pr,rc,_=precision_recall_curve(yte,p); ax[0].plot(rc,pr); ax[0].axhline(yte.mean(),ls='--',c='gray'); ax[0].set(xlabel='recall',ylabel='precision',title='Precision-recall')
fr,mp=calibration_curve(yte,p,n_bins=8,strategy='quantile'); ax[1].plot(mp,fr,'o-'); ax[1].plot([0,1],[0,1],'--',c='gray'); ax[1].set(xlabel='predicted',ylabel='observed',title='Calibration')
imp=permutation_importance(models[best],Xte,yte,scoring='roc_auc',n_repeats=20,random_state=0)
s=pd.Series(imp.importances_mean,index=feats).sort_values(); ax[2].barh(s.index,s.values); ax[2].set_title('Permutation importance (test AUC drop)')
plt.tight_layout(); plt.show()
o=np.argsort(-p); out=[]
for k in (.05,.10,.20,.30):
    n=int(len(o)*k); out.append(dict(flag_top=f'{k:.0%}',precision=y[o[:n]].mean(),recall=y[o[:n]].sum()/y.sum(),lift=y[o[:n]].mean()/y.mean()))
display(pd.DataFrame(out).round(3))

## Conclusions
* **Yes, partly.** Using only inspection-time measurements (vibration, bearing temperature, load, hours, ambient temperature, site), gradient boosting reaches test AUC ~0.80 (95% CI ~0.72-0.86) on the latest 20% of inspections, versus 0.50 for the prevalence baseline. Logistic regression is clearly weaker (~0.65), so the relationships are non-linear.
* **Precision is modest.** The test period has a 2.6% failure rate (31 failures), so flagging the top 10% of inspections catches ~39% of failures at ~10% precision (~4x lift). PR-AUC ~0.075 against a 0.026 random baseline.
* **`parts_replaced` is leakage** and was excluded; it would give a near-perfect but unusable model.
* **The process is non-stationary.** Monthly failure rates range from ~0% to ~39%, the train rate (14%) is far above the test rate (2.6%), and feature strength shifts between quarters. Forward-chaining CV inside the training period is much worse (~0.58 AUC for boosting) than shuffled CV (~0.85), partly because early folds train on little data from a different regime. Forward-looking performance is therefore uncertain; the ~0.80 figure rests on 31 test failures from one period.
* Predicted probabilities are not reliable in absolute terms (rate shift), so use the model to rank inspections rather than to quote failure probabilities, unless recalibrated on recent data.
* **Next steps:** find out why the failure rate varies over time (a missing driver such as an operating regime or maintenance campaign is likely); get per-asset history (each asset appears once here); confirm late-2025 labels have a complete 90-day follow-up window; retrain/recalibrate regularly.